# SatQuery GeoVision — Aerial YOLO Fine-Tuning Pipeline

This Google Colab notebook provides the standalone GPU training environment for the **SatQuery Aerial Object Detection Pipeline**.

### Pipeline Stages:
1. **Environment Setup & GPU Check** (T4 / A100)
2. **Aerial Dataset Preparation** (VisDrone & xView unified taxonomy)
3. **YOLOv8 Transfer Learning** (pretrained `yolov8m.pt` with aerial augmentations)
4. **Independent Validation & Metric Export** (Precision, Recall, F1, mAP50, mAP50-95)
5. **Export `best.pt` Checkpoint** for SatQuery AI deployment

In [ ]:
# 1. Verify GPU Availability
!nvidia-smi
!pip install -q ultralytics pyyaml pillow matplotlib

In [ ]:
# 2. Clone SatQuery Training Repository or Prepare Directories
import os
from pathlib import Path

WORKDIR = Path("/content/satquery_training")
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
print(f"Working directory: {WORKDIR}")

In [ ]:
# 3. Create SatQuery Aerial dataset.yaml configuration
dataset_yaml = """
path: /content/satquery_training/dataset
train: images/train
val: images/val
test: images/test

names:
  0: building
  1: house
  2: car
  3: truck
  4: bus
  5: motorcycle
  6: aircraft
  7: boat
  8: person
"""
with open("dataset.yaml", "w") as f:
    f.write(dataset_yaml.strip())
print("Created dataset.yaml with 9 SatQuery discrete aerial classes.")

In [ ]:
# 4. Launch YOLOv8 Transfer Learning on Aerial Dataset
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

results = model.train(
    data="dataset.yaml",
    epochs=100,
    batch=16,
    imgsz=640,
    lr0=0.001,
    patience=15,
    project="/content/satquery_training/runs",
    name="aerial_yolov8m",
    mosaic=1.0,
    mixup=0.15,
    degrees=10.0,
    fliplr=0.5,
    flipud=0.5,
    scale=0.5,
    save=True,
)
print("Training complete! Best weights at: /content/satquery_training/runs/aerial_yolov8m/weights/best.pt")

In [ ]:
# 5. Run Formal Evaluation on Independent Validation Split
trained_model = YOLO("/content/satquery_training/runs/aerial_yolov8m/weights/best.pt")
val_metrics = trained_model.val(data="dataset.yaml", split="val")

print("\n--- Verification Metrics ---")
print(f"Precision: {val_metrics.box.mp:.4f}")
print(f"Recall:    {val_metrics.box.mr:.4f}")
print(f"mAP50:     {val_metrics.box.map50:.4f}")
print(f"mAP50-95:  {val_metrics.box.map:.4f}")

In [ ]:
# 6. Export best.pt Checkpoint for SatQuery Deployment
from google.colab import files

best_weights = "/content/satquery_training/runs/aerial_yolov8m/weights/best.pt"
if os.path.exists(best_weights):
    print(f"Ready to download best.pt ({os.path.getsize(best_weights) / (1024*1024):.2f} MB)")
    files.download(best_weights)
else:
    print("Weights checkpoint not found.")